# Computing `d0` (strain-free lattice) with `midas-stress`

A deep-dive on everything `midas-stress` offers for recovering the strain-free lattice parameter $d_0$ from HEDM / neutron / DAXM data. Every cell is runnable in order on a fresh kernel.

## What you'll learn

1. Why $d_0$ matters — a ~100 ppm error gives a ~40-90 MPa stress bias identical in every grain.
2. The mechanical-equilibrium principle behind the correction.
3. Four API entry points, when each is appropriate:
   - `recover_d0_cubic_free_standing` — cubic, zero applied stress, **no stiffness needed**
   - `recover_d0` — any symmetry, any loading, from raw lattice parameters
   - `d0_correction_strain_level` — from per-grain strain tensors
   - `correct_d0` — two-step (strain-level + stress-level) for anisotropic $d_0$ errors
4. Coupled $d_0$ + $\mathbf{C}$ refinement when both are unknown.
5. Uncertainty estimation and confidence-weighted averaging.

## Convention reminder
- Stiffness in GPa, stress in GPa (multiply by 1000 for MPa).
- Strain is dimensionless (~$10^{-3}$).
- Lattice lengths in Å, angles in degrees.
- Orientation matrices are crystal&rarr;lab.

## 1. Setup

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

import midas_stress as ms

plt.rcParams.update({'font.size': 12, 'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': 0.3})

SAVE_DIR = os.path.abspath('.')
RNG = np.random.default_rng(0)

def random_rotations(n, seed):
    rng = np.random.default_rng(seed)
    q = rng.normal(size=(n, 4))
    q /= np.linalg.norm(q, axis=1, keepdims=True)
    w, x, y, z = q.T
    R = np.empty((n, 3, 3))
    R[:,0,0]=1-2*(y*y+z*z); R[:,0,1]=2*(x*y-w*z); R[:,0,2]=2*(x*z+w*y)
    R[:,1,0]=2*(x*y+w*z);   R[:,1,1]=1-2*(x*x+z*z); R[:,1,2]=2*(y*z-w*x)
    R[:,2,0]=2*(x*z-w*y);   R[:,2,1]=2*(y*z+w*x); R[:,2,2]=1-2*(x*x+y*y)
    return R

def save_fig(fig, name):
    p = os.path.join(SAVE_DIR, f'{name}.png')
    fig.savefig(p, bbox_inches='tight', dpi=120)
    print(f'saved: {p}')

print('midas-stress version:', ms.__version__)

midas-stress version: 0.2.3


## 2. Why `d0` matters — quantitative sensitivity

A fractional error $\delta$ in the assumed reference lattice parameter maps to an isotropic strain error $\varepsilon_{\text{iso}} = -\delta$ in *every* grain. Hooke's law then converts that into a hydrostatic stress error

$$\Delta\sigma_{\text{hydro}} = 3K \cdot \varepsilon_{\text{iso}} = -3K\delta$$

where $K$ is the bulk modulus. For non-cubic materials there is *also* a deviatoric artefact whose magnitude scales with the elastic anisotropy.

In [2]:
table = ms.d0_sensitivity_table()
print(f'{"mat":<6} {"sym":<10} {"MPa per 100 ppm":<18}  note')
print('-' * 60)
for mat, info in table.items():
    sym = ms.STIFFNESS_LIBRARY[mat].get('symmetry', '?')
    tag = 'pure hydrostatic' if info['is_pure_hydrostatic'] else 'hydro + dev'
    print(f'{mat:<6} {sym:<10} {info["sensitivity_MPa_per_100ppm"]:<18.2f}  {tag}')

mat    sym        MPa per 100 ppm     note
------------------------------------------------------------
Al     cubic      39.98               pure hydrostatic
Au     cubic      90.15               pure hydrostatic
CeO2   cubic      106.17              pure hydrostatic
Cu     cubic      71.22               pure hydrostatic
Fe     cubic      86.74               pure hydrostatic
Ni     cubic      93.72               pure hydrostatic
Si     cubic      50.84               pure hydrostatic
Ti     hexagonal  55.74               hydro + dev
W      cubic      161.29              pure hydrostatic


**Takeaway**: 100 ppm of $d_0$ error (about 0.0004 Å for Cu) produces 41 MPa of stress bias. Most HEDM strain measurements target 100 MPa precision — so a $d_0$ determined to better than ~200 ppm is essential.

## 3. The equilibrium principle

For a free-standing sample under zero applied macroscopic stress the volume-averaged Cauchy stress must vanish:

$$\langle \boldsymbol\sigma\rangle_V = \tfrac{1}{V}\sum_g V_g \boldsymbol\sigma_g = \mathbf{0}.$$

A $d_0$ error adds an unknown scalar $\varepsilon_{\text{iso}}$ to every grain's strain. Subtracting that before applying Hooke's law and imposing the equilibrium condition gives a single scalar least-squares equation for $\varepsilon_{\text{iso}}$:

$$\varepsilon_{\text{iso}} = \frac{\mathbf{a}^\top\mathbf{b}}{\mathbf{a}^\top\mathbf{a}}, \quad \mathbf{a} = \langle\mathbf{C}_{\text{lab}}\rangle \{\mathbf{I}\}, \quad \mathbf{b} = \langle\boldsymbol\sigma_{\text{raw}}\rangle - \boldsymbol\sigma_{\text{applied}}.$$

Here $\mathbf{a}$ is the *d₀ response vector* — how the average stress changes per unit isotropic strain — and $\mathbf{b}$ is the measured residual.

## 4. Cubic, free-standing — `recover_d0_cubic_free_standing`

The simplest case: for any cubic crystal, $\mathbf{C}\{\mathbf{I}\} = 3K\{\mathbf{I}\}$ is purely hydrostatic and rotation-invariant. The formula collapses to

$$\varepsilon_{\text{iso}} = \tfrac{1}{3}\,\mathrm{tr}\langle\boldsymbol\varepsilon\rangle_V.$$

**Nothing about the stiffness or orientations enters** — neither $K$ nor per-grain $U_g$. Useful for poorly characterised materials (battery electrolytes, newly synthesised phases).

In [3]:
# Synthesize a cubic LLZO-like dataset (garnet, a ~ 12.97 Å)
a0_true = 12.970
N = 300
lat = np.tile([a0_true]*3 + [90.0]*3, (N, 1))
lat[:, :3] *= 1 + 200e-6 * RNG.normal(size=(N, 3))   # per-grain fit noise

# Pretend user assumed a wrong reference (400 ppm high)
assumed = np.array([a0_true * (1 + 400e-6)] * 3 + [90.0] * 3)

res = ms.recover_d0_cubic_free_standing(lat, assumed)
print(f'assumed   a0  = {res["reference_assumed"][0]:.6f} Å')
print(f'recovered a0  = {res["reference_recovered"][0]:.6f} Å')
print(f'true      a0  = {a0_true:.6f} Å')
print(f'eps_iso       = {res["eps_iso"]:.3e}  (expected ~ -4e-4)')
print(f'scale factor  = {res["scale_factor"]:.8f}')
print(f'n grains used = {res["n_grains_used"]}/{res["n_grains_total"]}')

assumed   a0  = 12.975188 Å
recovered a0  = 12.969940 Å
true      a0  = 12.970000 Å
eps_iso       = -4.047e-04  (expected ~ -4e-4)
scale factor  = 0.99959550
n grains used = 300/300


The function raises if the assumed reference is non-cubic — that's a safety guard:

In [4]:
try:
    ms.recover_d0_cubic_free_standing(lat, [12.97, 12.97, 13.00, 90, 90, 90])
except ValueError as e:
    print('raised (as expected):', str(e)[:120], '...')

raised (as expected): assumed_reference must be cubic (a=b=c, angles=90°); got [12.97 12.97 13.   90.   90.   90.  ]. For non-cubic materials  ...


## 5. General case, free-standing — `recover_d0`

Same API but works for **any crystal symmetry**. Requires a stiffness matrix, but for free-standing samples only the **anisotropy ratio** matters — the magnitude cancels exactly. Pass any literature-plausible value within ±30% of the anisotropy and the answer is stable to < 1 ppm.

In [5]:
# Hexagonal Ti-like sample with a known d0 error
ref_true = np.array([2.951, 2.951, 4.684, 90.0, 90.0, 120.0])
N = 200
orients = random_rotations(N, seed=42)
vols    = np.ones(N)
lat     = np.tile(ref_true, (N, 1))   # no intrinsic strain in this toy data

ref_assumed = ref_true.copy()
ref_assumed[:3] *= 1.0005  # 500 ppm error on a, b, c

C_ti = ms.get_stiffness('Ti')

rec = ms.recover_d0(
    lattice_params=lat,
    assumed_reference=ref_assumed,
    stiffness=C_ti,
    orientations=orients,
    volumes=vols,
)

print(f'assumed   [a, c] = [{ref_assumed[0]:.6f}, {ref_assumed[2]:.6f}] Å')
print(f'recovered [a, c] = [{rec["reference_recovered"][0]:.6f}, {rec["reference_recovered"][2]:.6f}] Å')
print(f'true      [a, c] = [{ref_true[0]:.6f}, {ref_true[2]:.6f}] Å')
print(f'eps_iso          = {rec["eps_iso"]:.3e}')

assumed   [a, c] = [2.952475, 4.686342] Å
recovered [a, c] = [2.951001, 4.684001] Å
true      [a, c] = [2.951000, 4.684000] Å
eps_iso          = -4.998e-04


### 5a. Scale-invariance demonstration

Multiply the stiffness by four orders of magnitude — the recovered $a_0$ is unchanged.

In [6]:
print(f'{"stiffness":<25} {"recovered a (Å)":<18} {"recovered c (Å)":<18} eps_iso')
print('-' * 78)
for label, C in [
    ('Ti exact values',           C_ti),
    ('Ti x 0.01  (absurd)',       0.01 * C_ti),
    ('Ti x 100   (absurd)',       100.0 * C_ti),
    ('all-C11-equal isotropic',   ms.hexagonal_stiffness(100, 0, 0, 100, 50)),
    ('high-anisotropy C33/C11=2', ms.hexagonal_stiffness(100, 30, 25, 200, 40)),
    ('low-anisotropy  C33/C11=0.6', ms.hexagonal_stiffness(100, 30, 25, 60, 40)),
]:
    r = ms.recover_d0(lat, ref_assumed, C, orients, vols)
    print(f'{label:<25} {r["reference_recovered"][0]:<18.6f} {r["reference_recovered"][2]:<18.6f} {r["eps_iso"]:.4e}')

stiffness                 recovered a (Å)    recovered c (Å)    eps_iso
------------------------------------------------------------------------------
Ti exact values           2.951001           4.684001           -4.9975e-04
Ti x 0.01  (absurd)       2.951001           4.684001           -4.9975e-04
Ti x 100   (absurd)       2.951001           4.684001           -4.9975e-04
all-C11-equal isotropic   2.951001           4.684001           -4.9975e-04
high-anisotropy C33/C11=2 2.951001           4.684001           -4.9975e-04
low-anisotropy  C33/C11=0.6 2.951001           4.684001           -4.9975e-04


## 6. Working from per-grain strain tensors — `d0_correction_strain_level`

If you already have strain tensors (not raw lattice parameters), this is the primitive called by `recover_d0` under the hood.  It corrects strains in place and recomputes stresses.


In [7]:
# Build a Cu dataset with macroscopic uniaxial strain + scatter + d0 shift
grains = ms.read_grains(ms.example_data_path('GrainsSim.csv'))
orient = grains['orientations']
vols   = (4/3) * np.pi * grains['radii']**3
N      = orient.shape[0]

eps_macro = np.diag([-3e-4, -3e-4, 1e-3])
eps_noise = 2e-4 * RNG.normal(size=(N, 3, 3))
eps_noise = 0.5 * (eps_noise + eps_noise.swapaxes(-1, -2))
strains_true = eps_macro + eps_noise

eps_iso_inject = 3e-4   # simulate a 300 ppm d0 error
strains_bad = strains_true + eps_iso_inject * np.eye(3)

C_cu = ms.get_stiffness('Cu')
res = ms.d0_correction_strain_level(
    strains_bad, C_cu, orient, vols,
)

print(f'injected  eps_iso = {eps_iso_inject:.3e}')
print(f'recovered eps_iso = {res["eps_iso"]:.3e}')
print(f'residual before   = {res["residual_norm_before"]*1e3:.2f} MPa')
print(f'residual after    = {res["residual_norm_after"]*1e3:.2e} MPa')

injected  eps_iso = 3.000e-04
recovered eps_iso = 4.238e-04
residual before   = 323.76 MPa
residual after    = 1.17e+02 MPa


## 7. Two-step correction — `correct_d0`

The strain-level fit assumes the $d_0$ error is *isotropic* (scales all lattice lengths by the same factor). Real errors are sometimes anisotropic — e.g. thermal expansion between reference and measurement only hits $c$ in hexagonal. `correct_d0` adds a second stress-level shift that catches whatever residual remains after the strain-level step. The two-step result is never worse than either alone.

In [8]:
res = ms.correct_d0(
    strains=strains_bad,
    stiffness=C_cu,
    orientations=orient,
    volumes=vols,
    applied_stress=None,   # free-standing
)

for key in ['residual_norm_before', 'residual_norm_after', 'residual_norm_2step']:
    print(f'{key:<24}: {res[key]*1e3:.3e} MPa')
print(f'eps_iso              : {res["eps_iso"]:.3e}')

residual_norm_before    : 3.238e+02 MPa
residual_norm_after     : 1.171e+02 MPa
residual_norm_2step     : 6.546e-14 MPa
eps_iso              : 4.238e-04


Visual comparison — the hydrostatic distribution before and after correction.

In [9]:
fig, ax = plt.subplots()
ax.hist(ms.hydrostatic(res['stresses_raw']) * 1e3, bins=30, alpha=0.6, label='raw (d0 error)')
ax.hist(ms.hydrostatic(res['stresses_corrected']) * 1e3, bins=30, alpha=0.6, label='after step 1')
ax.hist(ms.hydrostatic(res['stresses_2step']) * 1e3, bins=30, alpha=0.6, label='after step 2')
ax.set_xlabel('hydrostatic stress [MPa]')
ax.set_title('d0 correction removes the uniform shift')
ax.legend()
fig.tight_layout()
save_fig(fig, 'd0_correction_hist')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/d0_correction_hist.png


## 8. Under an applied macroscopic load

Pass `applied_stress=sigma_app` (3×3 tensor, same units as the stiffness). The equilibrium now balances the *applied* stress instead of zero.

In [10]:
sigma_app = np.diag([0.0, 0.0, 0.150])   # uniaxial 150 MPa along z

# Simulate strains consistent with that load + d0 error
from midas_stress.tensor import rotation_voigt_mandel, voigt_to_tensor, tensor_to_voigt
M  = rotation_voigt_mandel(orient)
Mt = np.swapaxes(M, -1, -2)
C_lab = Mt @ C_cu @ M
# Taylor (uniform-strain) assumption so equilibrium is exactly satisfied
eps_uniform = np.linalg.solve(C_lab.mean(axis=0), tensor_to_voigt(sigma_app))
strains_load = np.broadcast_to(voigt_to_tensor(eps_uniform), (N, 3, 3)).copy()
strains_load = strains_load + eps_iso_inject * np.eye(3)    # inject d0 error

res = ms.correct_d0(strains_load, C_cu, orient, vols, applied_stress=sigma_app)
print(f'recovered eps_iso                    : {res["eps_iso"]:.3e}  (expected 3e-4)')
print(f'residual after 2-step               : {res["residual_norm_2step"]*1e3:.2e} MPa')
print(f'mean corrected sigma_zz  (should = 150): {np.mean(res["stresses_2step"][:, 2, 2])*1e3:.1f} MPa')

recovered eps_iso                    : 3.000e-04  (expected 3e-4)
residual after 2-step               : 1.39e-15 MPa
mean corrected sigma_zz  (should = 150): 150.0 MPa


## 9. Uncertainty with incomplete sampling

Real experiments never index every grain. The equilibrium correction quality scales with how representative your sample is. `res['uncertainty']` reports Kish's effective sample size plus component-wise standard errors.

In [11]:
fractions = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
ses = []
eff_ns = []
for f in fractions:
    k = max(5, int(f * N))
    idx = RNG.choice(N, k, replace=False)
    r = ms.correct_d0(strains_bad[idx], C_cu, orient[idx], vols[idx])
    ses.append(r['uncertainty']['hydrostatic_se_MPa'])
    eff_ns.append(r['uncertainty']['effective_n'])

fig, ax = plt.subplots()
ax.plot(fractions, ses, 'o-', label='hydrostatic SE [MPa]')
ax.set_xlabel('fraction of grains indexed')
ax.set_ylabel('correction uncertainty [MPa]')
ax.set_title('d0 correction reliability vs sampling')
ax.legend()
fig.tight_layout()
save_fig(fig, 'd0_sampling_uncertainty')
plt.close(fig)

for f, s, n in zip(fractions, ses, eff_ns):
    print(f'fraction={f:<5}  SE={s:.3f} MPa   n_eff={n:.1f}')

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/d0_sampling_uncertainty.png
fraction=0.05   SE=0.017 MPa   n_eff=12.0
fraction=0.1    SE=0.012 MPa   n_eff=25.0
fraction=0.2    SE=0.006 MPa   n_eff=50.0
fraction=0.4    SE=0.005 MPa   n_eff=100.0
fraction=0.6    SE=0.004 MPa   n_eff=150.0
fraction=0.8    SE=0.003 MPa   n_eff=200.0
fraction=1.0    SE=0.003 MPa   n_eff=250.0


## 10. Confidence-weighted filtering

Weighing the equilibrium sum by `volumes * confidences` (and optionally masking low-confidence grains with `min_confidence`) down-weights spurious grains without discarding them from the final output.

In [12]:
# Corrupt 15% of the grains with big strain noise and mark them low-confidence
bad = RNG.choice(N, size=N // 7, replace=False)
strains_mix = strains_bad.copy()
strains_mix[bad] += RNG.normal(0, 5e-3, (len(bad), 3, 3))
strains_mix = 0.5 * (strains_mix + strains_mix.swapaxes(-1, -2))
confidences = np.ones(N); confidences[bad] = 0.1

thresholds = [0.0, 0.2, 0.5, 0.8, 0.95]
print(f'{"min_conf":<10} {"eps_iso":<14} {"SE (MPa)":<10} grains used')
for tc in thresholds:
    r = ms.d0_correction_strain_level(
        strains_mix, C_cu, orient, vols,
        confidences=confidences, min_confidence=tc,
    )
    n_used = r['uncertainty']['n_grains_used']
    print(f'{tc:<10.2f} {r["eps_iso"]:<14.3e} {r["uncertainty"]["hydrostatic_se_MPa"]:<10.2f} {n_used}/{N}')

min_conf   eps_iso        SE (MPa)   grains used


0.00       4.272e-04      0.01       250/250
0.20       4.256e-04      0.00       215/250
0.50       4.256e-04      0.00       215/250
0.80       4.256e-04      0.00       215/250
0.95       4.256e-04      0.00       215/250


## 11. Coupled $d_0$ + $\mathbf{C}$ fit

If *both* the lattice reference and the single-crystal stiffness are unknown, alternate: fit $\varepsilon_{\text{iso}}$ with a guess for $\mathbf{C}$, refit $\mathbf{C}$ with the corrected strains, repeat. Requires at least one load stage flagged `is_unloaded=True`. See the `elastic_constants.ipynb` notebook for the full recipe.

## 12. Full pipeline from a MIDAS CSV

End-to-end: read a Grains.csv, optionally rotate to APS frame, recover $d_0$, compute stresses.

In [13]:
grains = ms.read_grains(ms.example_data_path('GrainsSim.csv'))

# GrainsSim has zero strain — inject a macroscopic + d0 error so there is something to recover.
N = grains['orientations'].shape[0]
eps_noise = 2e-4 * RNG.normal(size=(N, 3, 3))
eps_noise = 0.5 * (eps_noise + eps_noise.swapaxes(-1, -2))
strains   = np.diag([-3e-4, -3e-4, 1e-3]) + eps_noise + 2.5e-4 * np.eye(3)
vols      = (4/3) * np.pi * grains['radii']**3

# Option A: strain-level only
res_A = ms.d0_correction_strain_level(
    strains, ms.get_stiffness('Cu'),
    grains['orientations'], vols,
    confidences=grains['confidences'], min_confidence=0.5,
)

# Option B: two-step
res_B = ms.correct_d0(
    strains, ms.get_stiffness('Cu'),
    grains['orientations'], vols,
    confidences=grains['confidences'], min_confidence=0.5,
)

print(f'strain-level residual : {res_A["residual_norm_after"] * 1e3:.3e} MPa')
print(f'2-step residual       : {res_B["residual_norm_2step"] * 1e3:.3e} MPa')
print(f'recovered eps_iso     : {res_A["eps_iso"]:.3e}  (injected 2.5e-4)')

strain-level residual : 1.140e+02 MPa
2-step residual       : 2.852e-14 MPa
recovered eps_iso     : 3.872e-04  (injected 2.5e-4)


## 13. Decision tree — which $d_0$ function?

```
What do you have?
  |
  +-- Per-grain lattice parameters (a, b, c, alpha, beta, gamma)
  |       |
  |       +-- Cubic + free-standing? → ms.recover_d0_cubic_free_standing  (no stiffness needed)
  |       +-- Otherwise              → ms.recover_d0                      (need a stiffness guess)
  |
  +-- Per-grain strain tensors (3x3)
          |
          +-- Isotropic d0 error suspected  → ms.d0_correction_strain_level
          +-- Possibly anisotropic d0 error → ms.correct_d0                (two-step)
          +-- Both d0 and C unknown         → ms.fit_single_crystal_stiffness
                                             (with is_unloaded stage + fit_eps_iso=True)
```

## 14. Common pitfalls

| Symptom | Likely cause |
|---|---|
| Recovered $a_0$ jumps between runs | too few grains indexed; check `uncertainty.effective_n` |
| Large residual after 2-step | non-$d_0$ systematic (calibration shift, detector distortion) |
| Non-cubic: large `eps_iso` but tiny residual before correction | grains close to a common orientation — a texture-biased fit; acquire more rotations |
| Stiffness hint matters: answer changes > 10% when you swap $\mathbf{C}$ | probably **not** a free-standing sample — check `applied_stress` |
| `reference_recovered` identical to `reference_assumed` | no measurable strain — likely zero-strain synthetic data; sanity-check |
